# 1、SummarizationMiddleware中间件

## 举例1：测试trigger、keep参数

In [10]:
import truststore
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from tencentcloud.common import profile

# 从.env文件中加载环境变量
load_dotenv(override=True)
truststore.inject_into_ssl()

# CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
# CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
#
# model = init_chat_model(
#     model="gpt-5.4-mini",
#     model_provider="openai",
#     api_key=CLOSEAI_API_KEY,
#     base_url=CLOSEAI_BASE_URL
# )
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    profile={
        "max_input_tokens": 1_000_000
    }
)

In [12]:

from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2)
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

#rprint(response)

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user introduced themselves as “老王” and the AI introduced itself as “小王”. The user expressed pleasure in meeting the AI. The primary goal is establishing a friendly introductory interaction; no concrete task was defined beyond initial social exchange.

## SUMMARY
- User’s name: 老王.
- AI’s chosen name: 小王.
- Tone: Friendly, casual greeting. No substantive task or project was discussed.

## ARTIFACTS
None.

## NEXT STEPS
Await user’s further instructions or requests to define a specific goal or task for the session.
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，老王别误会，我开玩笑呢！我的意思是——咱俩这刚认识，你还没来得及说正事儿呢，我就急着“高兴”上了，怕你其实有大事要交代，那我就

## 举例2：summary_prompt

In [13]:

from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.agents import create_agent


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]



agent = create_agent(
    model="deepseek-v4-flash",
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens",100),
                ("messages",6),
                ("fraction",0.001)
            ],
            keep=("messages",2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)

response = agent.invoke({
    "messages": messages
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================

Here is a summary of the conversation to date:

好的，这是对历史消息的摘要：

**对话参与者：**
- 用户：老王（人类）
- AI：小王（AI助手）

**对话内容摘要：**
1. 系统设定：AI助手被设定为“非常友好”。
2. 用户老王主动打招呼并自我介绍。
3. AI自我介绍为“小王”。
4. 用户老王表示很高兴认识小王，对话氛围友好融洽。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，老王别误会！我的意思是——咱俩刚认识就这么投缘，后面肯定还有更多有意思的话题可以聊，所以“高兴”这事儿啊，才刚开始呢！😄

我可没有泼冷水的意思。来来来，今天想聊点啥？
